# NBA Quant AI — GPU Evolution with TabICLv2 + TabPFN

**Goal**: Beat all-time Brier 0.21976 using GPU models + evolved features.

Population seeded from S10-S15 best individuals.

**BEFORE RUNNING**:
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all cells (Ctrl+F9)

In [ ]:
# ============================================================
# CELL 0: FIX CUDA + HF TOKEN (run this FIRST)
# ============================================================
import subprocess, sys, os, time, gc, json

# 1) Force CUDA PyTorch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu121'])

# 2) HF Token — from Colab secrets ONLY
# Setup: Runtime → Secrets (key icon) → Add HF_TOKEN
try:
    from google.colab import userdata
    _t = userdata.get('HF_TOKEN')
    if _t:
        os.environ['HF_TOKEN'] = _t
        os.environ['HUGGING_FACE_HUB_TOKEN'] = _t
        print('HF_TOKEN: from Colab secrets')
    else:
        raise ValueError('HF_TOKEN secret is empty')
except Exception as e:
    print(f'ERROR: {e}')
    print('You MUST set HF_TOKEN in Colab secrets:')
    print('  → Click the key icon in the left sidebar')
    print('  → Add secret: Name=HF_TOKEN, Value=hf_...')
    print('  → Toggle "Notebook access" ON')

# 3) Database URL (for logging)
try:
    db_url = userdata.get('DATABASE_URL')
    if db_url:
        os.environ['DATABASE_URL'] = db_url
        print('DATABASE_URL: from Colab secrets')
except Exception:
    print('DATABASE_URL: not set (logging disabled)')

# Verify CUDA
import torch
print(f'\nPyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
else:
    print('WARNING: CUDA not available! Go to Runtime → Change runtime type → GPU (T4)')

In [ ]:
# ============================================================
# CELL 1: CLONE REPO + INSTALL DEPS
# ============================================================
if not os.path.exists('/content/nomos-nba-agent'):
    subprocess.run(['git', 'clone', 'https://github.com/LBJLincoln/nomos-nba-agent.git',
                    '/content/nomos-nba-agent'], check=True)
else:
    subprocess.run(['git', '-C', '/content/nomos-nba-agent', 'pull'], check=True)

os.chdir('/content/nomos-nba-agent')
sys.path.insert(0, '/content/nomos-nba-agent')

# Install ML deps AFTER torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'psycopg2-binary', 'xgboost', 'lightgbm', 'catboost',
    'scikit-learn>=1.3', 'pandas', 'numpy', 'scipy', 'nba_api',
    'tabicl', 'tabpfn', 'huggingface_hub'])

# HF login
try:
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HuggingFace login: OK')
except Exception as e:
    print(f'HF login failed: {e}')

# Re-verify CUDA (pip install can break it)
import importlib; importlib.reload(torch)
if not torch.cuda.is_available():
    print('CUDA broken by pip! Fixing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        'torch', '--index-url', 'https://download.pytorch.org/whl/cu121'])
    importlib.reload(torch)
print(f'CUDA: {torch.cuda.is_available()}')

# Pre-download model checkpoints
import numpy as np
print('\nPre-downloading checkpoints...')
for name, code in [('TabPFN', 'from tabpfn import TabPFNClassifier; m=TabPFNClassifier(); m.fit(np.random.randn(50,5), np.random.randint(0,2,50))'),
                    ('TabICLv2', 'from tabicl import TabICLClassifier; m=TabICLClassifier(); m.fit(np.random.randn(50,5), np.random.randint(0,2,50))')]:
    try:
        exec(code)
        print(f'  {name}: cached OK')
        del m
    except Exception as e:
        print(f'  {name}: FAILED ({e})')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 2. Build Features + Seed Population from 6 Islands

In [ ]:
from pathlib import Path

# ---- Build features ----
CACHE_FILE = Path('/content/features_cache.npz')

if CACHE_FILE.exists():
    print('Loading cached features...')
    cached = np.load(CACHE_FILE, allow_pickle=True)
    X = cached['X']
    y = cached['y']
    feature_names = cached['feature_names'].tolist()
else:
    print('Building features (~30 min, will cache)...')
    data_dir = Path('/content/nomos-nba-agent/hf-space/data/historical')
    games = []
    for f in sorted(data_dir.glob('games-*.json')):
        raw = json.loads(f.read_text())
        if isinstance(raw, list): games.extend(raw)
        elif isinstance(raw, dict) and 'games' in raw: games.extend(raw['games'])
    print(f'Games: {len(games)}')

    sys.path.insert(0, '/content/nomos-nba-agent/hf-space')
    from features.engine import NBAFeatureEngine
    engine = NBAFeatureEngine()
    X, y, feature_names = engine.build(games)
    X = np.nan_to_num(np.array(X, dtype=np.float64))
    y = np.array(y, dtype=np.int32)
    variances = np.var(X, axis=0)
    valid = variances > 1e-10
    X = X[:, valid]
    feature_names = [f for f, v in zip(feature_names, valid) if v]
    np.savez_compressed(CACHE_FILE, X=X, y=y, feature_names=np.array(feature_names))

n_features = X.shape[1]
print(f'Features: {X.shape} ({n_features} usable)')

# ---- Fetch seeds from 6 islands ----
import requests
ISLANDS = {
    'S10': 'https://lbjlincoln-nomos-nba-quant.hf.space',
    'S11': 'https://lbjlincoln-nomos-nba-quant-2.hf.space',
    'S12': 'https://lbjlincoln26-nba-evo-3.hf.space',
    'S13': 'https://lbjlincoln26-nba-evo-4.hf.space',
    'S14': 'https://nomos42-nba-evo-5.hf.space',
    'S15': 'https://nomos42-nba-evo-6.hf.space',
}

island_seeds = []
print('\nFetching seeds from islands...')
for name, url in ISLANDS.items():
    try:
        resp = requests.get(f'{url}/api/results', timeout=15)
        if resp.status_code != 200:
            print(f'  {name}: HTTP {resp.status_code}')
            continue
        data = resp.json()
        best = data.get('best', {})
        island_seeds.append({
            'source': name,
            'brier': best.get('brier', 1.0),
            'features': best.get('selected_features', []),
            'model_type': best.get('model_type', 'xgboost'),
        })
        print(f'  {name}: brier={best.get("brier","?"):.5f} model={best.get("model_type","?")} n={best.get("n_features","?")}')
        # Also grab top3
        for i, ind in enumerate(data.get('top5', [])[:3]):
            island_seeds.append({
                'source': f'{name}_top{i+1}',
                'brier': ind.get('brier', 1.0),
                'features': ind.get('selected_features', []),
                'model_type': ind.get('model_type', 'xgboost'),
            })
    except Exception as e:
        print(f'  {name}: {e}')

print(f'Total seeds: {len(island_seeds)}')

## 3. Run GPU Evolution

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
import random
from collections import Counter
from copy import deepcopy

# ---- Config ----
POP_SIZE = 80
ELITE_SIZE = 8
N_SPLITS = 5
PURGE_GAP = 5
TARGET_FEATURES = 80
MAX_FEATURES = 200
MUTATION_RATE = 0.10
CROSSOVER_RATE = 0.80
MUT_FLOOR = 0.05
MUT_DECAY = 0.998
TOTAL_GENS = 500

_xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_tabpfn_device = 'cuda' if torch.cuda.is_available() else 'cpu'

GPU_MODEL_TYPES = ['xgboost', 'lightgbm', 'random_forest', 'extra_trees', 'catboost', 'tabicl', 'tabpfn']

# Check which GPU models are actually available
_available_models = ['xgboost', 'lightgbm', 'random_forest', 'extra_trees']
try:
    import catboost; _available_models.append('catboost')
except: pass
try:
    from tabicl import TabICLClassifier; _available_models.append('tabicl')
except: pass
try:
    from tabpfn import TabPFNClassifier; _available_models.append('tabpfn')
except: pass

GPU_MODEL_TYPES = _available_models
print(f'Available models: {GPU_MODEL_TYPES}')
print(f'XGBoost device: {_xgb_device}')

# ---- Walk-forward splitter ----
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
splits = [(tr[:-PURGE_GAP] if len(tr) > PURGE_GAP + 50 else tr, te)
          for tr, te in tscv.split(X)]

def make_model(model_type, hp):
    if model_type == 'xgboost':
        return xgb.XGBClassifier(
            n_estimators=hp.get('n_estimators', 200),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            reg_alpha=hp.get('reg_alpha', 0.01),
            reg_lambda=hp.get('reg_lambda', 1.0),
            tree_method='hist', device=_xgb_device,
            random_state=42, verbosity=0)
    elif model_type == 'lightgbm':
        return lgb.LGBMClassifier(
            n_estimators=hp.get('n_estimators', 200),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            reg_alpha=hp.get('reg_alpha', 0.01),
            reg_lambda=hp.get('reg_lambda', 1.0),
            random_state=42, verbose=-1)
    elif model_type == 'random_forest':
        return RandomForestClassifier(
            n_estimators=hp.get('n_estimators', 200),
            max_depth=hp.get('max_depth', 8),
            min_samples_leaf=5, random_state=42, n_jobs=-1)
    elif model_type == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=hp.get('n_estimators', 200),
            max_depth=hp.get('max_depth', 8),
            min_samples_leaf=5, random_state=42, n_jobs=-1)
    elif model_type == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=hp.get('n_estimators', 200),
            depth=min(hp.get('max_depth', 6), 10),
            learning_rate=hp.get('learning_rate', 0.05),
            random_state=42, verbose=0)
    elif model_type == 'tabicl':
        from tabicl import TabICLClassifier
        return TabICLClassifier()
    elif model_type == 'tabpfn':
        from tabpfn import TabPFNClassifier
        return TabPFNClassifier(device=_tabpfn_device)
    else:
        return ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)

def evaluate(features_mask, model_type, hp):
    """Evaluate an individual via walk-forward CV. Returns brier score."""
    indices = [i for i, v in enumerate(features_mask) if v]
    if len(indices) < 5: return 1.0
    if len(indices) > MAX_FEATURES: indices = indices[:MAX_FEATURES]
    X_sub = X[:, indices]
    X_sub = np.nan_to_num(X_sub, nan=0.0, posinf=1e6, neginf=-1e6)

    fold_briers = []
    for train_idx, test_idx in splits:
        try:
            model = make_model(model_type, hp)
            model.fit(X_sub[train_idx], y[train_idx])
            probs = model.predict_proba(X_sub[test_idx])[:, 1]
            fold_briers.append(brier_score_loss(y[test_idx], probs))
            del model
        except Exception:
            fold_briers.append(0.30)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return float(np.mean(fold_briers))

# ---- Individual class ----
class Individual:
    def __init__(self, features=None, model_type=None, hp=None):
        if features is None:
            prob = TARGET_FEATURES / max(n_features, 1)
            self.features = [1 if random.random() < prob else 0 for _ in range(n_features)]
        else:
            self.features = list(features)
        self.model_type = model_type or random.choice(GPU_MODEL_TYPES)
        self.hp = hp or {
            'n_estimators': random.randint(100, 400),
            'max_depth': random.randint(4, 9),
            'learning_rate': 10 ** random.uniform(-2.0, -0.7),
            'subsample': random.uniform(0.6, 1.0),
            'colsample_bytree': random.uniform(0.4, 1.0),
            'reg_alpha': 10 ** random.uniform(-4, 0),
            'reg_lambda': 10 ** random.uniform(-4, 0),
        }
        self.brier = 1.0
        self.n_feat = sum(self.features)
        self._enforce_cap()

    def _enforce_cap(self):
        indices = [i for i, v in enumerate(self.features) if v]
        if len(indices) > MAX_FEATURES:
            drop = random.sample(indices, len(indices) - MAX_FEATURES)
            for i in drop: self.features[i] = 0
        self.n_feat = sum(self.features)

    def eval(self):
        self.brier = evaluate(self.features, self.model_type, self.hp)
        return self.brier

    def mutate(self, rate):
        for i in range(len(self.features)):
            if random.random() < rate:
                self.features[i] = 1 - self.features[i]
        # Mutate hyperparams
        if random.random() < 0.3:
            self.hp['n_estimators'] = max(50, self.hp['n_estimators'] + random.randint(-50, 50))
        if random.random() < 0.3:
            self.hp['max_depth'] = max(2, min(12, self.hp['max_depth'] + random.randint(-2, 2)))
        if random.random() < 0.3:
            self.hp['learning_rate'] *= 10 ** random.uniform(-0.3, 0.3)
            self.hp['learning_rate'] = max(0.001, min(0.5, self.hp['learning_rate']))
        # Occasionally switch model type
        if random.random() < 0.10:
            self.model_type = random.choice(GPU_MODEL_TYPES)
        self._enforce_cap()
        self.brier = 1.0  # needs re-evaluation

    @staticmethod
    def crossover(p1, p2):
        point = random.randint(1, len(p1.features) - 1)
        child_feat = p1.features[:point] + p2.features[point:]
        child_hp = {k: p1.hp[k] if random.random() < 0.5 else p2.hp[k] for k in p1.hp}
        child_mt = p1.model_type if random.random() < 0.5 else p2.model_type
        child = Individual(features=child_feat, model_type=child_mt, hp=child_hp)
        return child

# ---- Create seeded population ----
name_to_idx = {name: i for i, name in enumerate(feature_names)}
population = []

for seed in island_seeds:
    feat = [0] * n_features
    for fname in seed.get('features', []):
        if isinstance(fname, str) and fname in name_to_idx:
            feat[name_to_idx[fname]] = 1
    if sum(feat) < 15:
        prob = TARGET_FEATURES / max(n_features, 1)
        feat = [1 if random.random() < prob else 0 for _ in range(n_features)]
    ind = Individual(features=feat, model_type=seed.get('model_type', 'xgboost'))
    population.append(ind)
    # TabICLv2 variant
    if 'tabicl' in GPU_MODEL_TYPES and len(population) < POP_SIZE:
        ind2 = Individual(features=list(feat), model_type='tabicl')
        population.append(ind2)

# Fill remaining with random
while len(population) < POP_SIZE:
    population.append(Individual())
population = population[:POP_SIZE]

mt_counts = Counter(ind.model_type for ind in population)
print(f'Population: {len(population)} individuals')
print(f'Model types: {dict(mt_counts)}')

# ---- State file for surviving disconnects ----
STATE_FILE = Path('/content/evolution_state.json')
best_ever_brier = 1.0
best_ever_info = None
start_gen = 0

if STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text())
        start_gen = state.get('generation', 0)
        best_ever_brier = state.get('best_brier', 1.0)
        best_ever_info = state.get('best_info')
        print(f'Restored state: gen={start_gen}, best={best_ever_brier:.5f}')
    except:
        pass

In [ ]:
# ============================================================
# CELL 4: EVOLUTION LOOP
# ============================================================
print('='*70)
print('  NBA QUANT AI — GPU EVOLUTION')
print(f'  Pop={POP_SIZE} | Gens={TOTAL_GENS} | Folds={N_SPLITS} | Models={len(GPU_MODEL_TYPES)}')
print('='*70)

mut_rate = MUTATION_RATE

for gen in range(start_gen, TOTAL_GENS):
    gen_start = time.time()

    # Evaluate unevaluated individuals
    n_eval = 0
    for ind in population:
        if ind.brier >= 0.99:
            ind.eval()
            n_eval += 1

    # Sort by brier (lower is better)
    population.sort(key=lambda x: x.brier)

    # Track best ever
    gen_best = population[0]
    improved = False
    if gen_best.brier < best_ever_brier:
        best_ever_brier = gen_best.brier
        best_ever_info = {
            'brier': gen_best.brier,
            'model_type': gen_best.model_type,
            'n_features': gen_best.n_feat,
            'features': [feature_names[i] for i, v in enumerate(gen_best.features) if v],
            'hp': gen_best.hp,
            'generation': gen,
        }
        improved = True

    gen_dur = time.time() - gen_start
    marker = ' *** NEW BEST ***' if improved else ''
    print(f'Gen {gen+1}/{TOTAL_GENS}: best={gen_best.brier:.5f} ({gen_best.model_type}, {gen_best.n_feat}f) '
          f'| ever={best_ever_brier:.5f} | {n_eval} evals {gen_dur:.0f}s{marker}')

    # Log every 10 gens
    if (gen + 1) % 10 == 0:
        mt = Counter(ind.model_type for ind in population)
        top5 = [(ind.brier, ind.model_type, ind.n_feat) for ind in population[:5]]
        print(f'  Models: {dict(mt)}')
        print(f'  Top 5: {top5}')

    # Save state every 25 gens
    if (gen + 1) % 25 == 0:
        STATE_FILE.write_text(json.dumps({
            'generation': gen + 1,
            'best_brier': best_ever_brier,
            'best_info': best_ever_info,
            'mutation_rate': mut_rate,
        }, default=str))

    # ---- Selection + Reproduction ----
    elite = population[:ELITE_SIZE]

    def tournament(pop, k=5):
        return min(random.sample(pop, min(k, len(pop))), key=lambda x: x.brier)

    children = [Individual(features=list(e.features), model_type=e.model_type, hp=dict(e.hp)) for e in elite]
    for e in children[:ELITE_SIZE]:
        e.brier = elite[children.index(e)].brier  # preserve elite fitness

    while len(children) < POP_SIZE:
        if random.random() < CROSSOVER_RATE:
            p1, p2 = tournament(population), tournament(population)
            child = Individual.crossover(p1, p2)
        else:
            child = Individual(features=list(tournament(population).features),
                             model_type=tournament(population).model_type,
                             hp=dict(tournament(population).hp))
        child.mutate(mut_rate)

        # Ensure GPU model representation
        if random.random() < 0.12:
            gpu_models = [m for m in ['tabicl', 'tabpfn'] if m in GPU_MODEL_TYPES]
            if gpu_models:
                child.model_type = random.choice(gpu_models)
        children.append(child)

    population = children[:POP_SIZE]

    # Adaptive mutation decay (capped at 0.15)
    mut_rate = max(MUT_FLOOR, min(0.15, mut_rate * MUT_DECAY))

# Final save
STATE_FILE.write_text(json.dumps({
    'generation': TOTAL_GENS,
    'best_brier': best_ever_brier,
    'best_info': best_ever_info,
    'mutation_rate': mut_rate,
}, default=str))

In [ ]:
# ============================================================
# CELL 5: FINAL RESULTS
# ============================================================
print('\n' + '='*70)
print('  FINAL RESULTS')
print('='*70)

if best_ever_info:
    print(f'  Best Brier:    {best_ever_info["brier"]:.5f}')
    print(f'  Model:         {best_ever_info["model_type"]}')
    print(f'  Features:      {best_ever_info["n_features"]}')
    print(f'  Generation:    {best_ever_info["generation"]}')

    # Top 5
    population.sort(key=lambda x: x.brier)
    print(f'\n  Top 5:')
    for i, ind in enumerate(population[:5]):
        print(f'    #{i+1}: brier={ind.brier:.5f} | {ind.model_type} | {ind.n_feat}f')

    # Model distribution in top 20
    mt_top20 = Counter(ind.model_type for ind in population[:20])
    print(f'\n  Model types in top 20: {dict(mt_top20)}')

    # Compare
    print(f'\n  All-time best (MOVDA-era): 0.22041')
    print(f'  All-time best (ever):      0.21976')
    delta = best_ever_info['brier'] - 0.21976
    print(f'  Delta vs all-time:         {delta:+.5f} ({"NEW RECORD!" if delta < 0 else "keep evolving"})')
else:
    print('  No results yet.')